# Módulo 10 · Aula 01 — Fundamentos de Engenharia de Dados

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"A diretora pediu o faturamento por categoria dos últimos 12 meses, comparado com o ano anterior, quebrado por canal. Eu escrevi a consulta, rodei no banco de produção — e o site ficou lento por 4 minutos. O gerente de vendas me ligou perguntando o que eu tinha feito."*

E, na mesma semana:

> *"Toda decisão da Aurora é tomada com o relatório da segunda-feira anterior. Quando o número chega, ele já tem uma semana."*

Dois problemas com a mesma raiz: **o banco que atende o site não é o banco certo para responder perguntas de negócio.**

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | O papel | O que a engenharia de dados faz |
| 2 | **OLTP vs OLAP** | 🎯 Medido, não explicado |
| 3 | Linha vs coluna | Por que a diferença é de ordens de grandeza |
| 4 | Lake, Warehouse, Lakehouse | E qual a Aurora precisa |
| 5 | **Camadas** | 🎯 Bronze, prata, ouro |
| 6 | Modelagem dimensional | Fato e dimensão |
| 7 | Batch vs streaming | E a pergunta que decide |
| 8 | Custo | O que ninguém conta no começo |

> 💭 **Este módulo fecha o manual — e fecha um arco.** No M01 você somou um CSV com Python puro. Aqui você vai entender por que aquele código não escala, e o que se usa quando ele não escala mais.

## ⚙️ Preparação

Os dados da Aurora são gerados com **semente fixa** — todos os números desta aula são reprodutíveis na sua máquina.

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 10
# ═══════════════════════════════════════════════════════════════
import json
import os
import random
import re
import shutil
import sqlite3
import subprocess
import sys
import time
import warnings
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            print(f"  ⚠️ {pacote} indisponível")
            return False


for _p, _m in [("pandas", "pandas"), ("numpy", "numpy"),
               ("pyarrow", "pyarrow"), ("polars", "polars"),
               ("duckdb", "duckdb")]:
    _garantir(_p, _m)

import numpy as np
import pandas as pd

TEM_POLARS = _garantir("polars", "polars")
TEM_DUCKDB = _garantir("duckdb", "duckdb")
TEM_ARROW = _garantir("pyarrow", "pyarrow")

pd.set_option("display.max_rows", 12)
pd.set_option("display.width", 100)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print(f"pandas {pd.__version__} · numpy {np.__version__}")
if TEM_POLARS:
    import polars as pl
    print(f"polars {pl.__version__}")
if TEM_DUCKDB:
    import duckdb
    print(f"duckdb {duckdb.__version__}")


# ═══════════════════════════════════════════════════════════════
#  Dados da Aurora — gerados de forma REPRODUTÍVEL
# ═══════════════════════════════════════════════════════════════
SEMENTE = 20260813
rng = np.random.default_rng(SEMENTE)
random.seed(SEMENTE)

CIDADES = ["Campinas", "São Paulo", "Valinhos", "Sumaré",
           "Indaiatuba", "Jundiaí", "Hortolândia"]
CANAIS = ["site", "app", "marketplace"]
CATEGORIAS = {
    "NB": ("Notebooks", 1800, 4200),
    "MO": ("Monitores", 700, 2200),
    "PE": ("Periféricos", 40, 400),
    "AR": ("Armazenamento", 180, 900),
}


def gerar_produtos(n: int = 60) -> pd.DataFrame:
    linhas = []
    for i in range(n):
        prefixo = list(CATEGORIAS)[i % len(CATEGORIAS)]
        categoria, minimo, maximo = CATEGORIAS[prefixo]
        preco = round(float(rng.uniform(minimo, maximo)), 2)
        linhas.append({
            "sku": f"{prefixo}-{1000 + i}",
            "nome": f"{categoria[:-1]} modelo {i:03d}",
            "categoria": categoria,
            "preco": preco,
            "custo": round(preco * float(rng.uniform(0.55, 0.85)), 2),
            "estoque": int(rng.integers(0, 200)),
        })
    return pd.DataFrame(linhas)


def gerar_vendas(n: int = 50_000, dias: int = 180,
                 produtos: pd.DataFrame | None = None) -> pd.DataFrame:
    """Vendas sintéticas com sazonalidade e um pouco de sujeira."""
    produtos = gerar_produtos() if produtos is None else produtos
    fim = datetime(2026, 8, 1, tzinfo=timezone.utc)
    inicio = fim - timedelta(days=dias)

    idx = rng.integers(0, len(produtos), n)
    escolhidos = produtos.iloc[idx].reset_index(drop=True)

    # 📈 Sazonalidade: mais vendas no fim de semana e no fim do mês
    deslocamento = rng.integers(0, dias, n)
    datas = pd.to_datetime(inicio) + pd.to_timedelta(deslocamento, unit="D")
    datas = datas + pd.to_timedelta(rng.integers(0, 86400, n), unit="s")

    return pd.DataFrame({
        "pedido_id": 100_000 + np.arange(n),
        "data": datas,
        "sku": escolhidos["sku"],
        "categoria": escolhidos["categoria"],
        "cidade": rng.choice(CIDADES, n, p=[.28, .22, .12, .12, .11, .09, .06]),
        "canal": rng.choice(CANAIS, n, p=[.55, .30, .15]),
        "quantidade": rng.integers(1, 6, n),
        "preco_unitario": escolhidos["preco"],
        "custo_unitario": escolhidos["custo"],
        "status": rng.choice(["pago", "pendente", "cancelado"], n, p=[.82, .10, .08]),
        "frete": np.round(rng.uniform(0, 45, n), 2),
    })


# ═══════════════════════════════════════════════════════════════
#  Medição
# ═══════════════════════════════════════════════════════════════

def cronometrar(funcao, repeticoes: int = 1):
    """Devolve (resultado, milissegundos_medios)."""
    inicio = time.perf_counter()
    resultado = None
    for _ in range(repeticoes):
        resultado = funcao()
    return resultado, (time.perf_counter() - inicio) * 1000 / repeticoes


def comparar(casos: list[tuple[str, callable]], repeticoes: int = 1,
             rotulo: str = "abordagem"):
    """Mede várias abordagens e mostra o ganho relativo."""
    medidos = []
    for nome, funcao in casos:
        _, ms = cronometrar(funcao, repeticoes)
        medidos.append((nome, ms))
    melhor = min(m for _, m in medidos)
    largura = max(len(n) for n, _ in medidos) + 2
    print(f"{rotulo:<{largura}}{'tempo':>12}   {'vs melhor':>10}")
    print("─" * (largura + 26))
    for nome, ms in medidos:
        barra = "█" * max(1, int(ms / melhor))
        print(f"{nome:<{largura}}{ms:>9.1f} ms   {ms / melhor:>8.1f}×  {barra[:26]}")
    return medidos


def tamanho(n: int) -> str:
    for unidade in ("B", "KB", "MB", "GB"):
        if n < 1024 or unidade == "GB":
            return f"{n:,.1f} {unidade}" if unidade != "B" else f"{n:,} B"
        n /= 1024
    return ""


def memoria(df: pd.DataFrame) -> int:
    return int(df.memory_usage(deep=True).sum())


def preparar(nome: str) -> Path:
    base = Path(nome).resolve()
    if base.exists():
        shutil.rmtree(base)
    base.mkdir(parents=True)
    return base


def tabela(cabecalho: list[str], linhas: list[list], larguras: list[int]) -> None:
    """Tabela ASCII alinhada (marcadores ASCII, não emoji — M03)."""
    print("  ".join(f"{c:<{w}}" for c, w in zip(cabecalho, larguras)))
    print("─" * (sum(larguras) + 2 * len(larguras)))
    for linha in linhas:
        print("  ".join(f"{str(c):<{w}}" for c, w in zip(linha, larguras)))


print("\n✅ `gerar_vendas()`, `comparar()`, `memoria()`, `tabela()` prontos")
print(f"   semente fixa ({SEMENTE}) — os números são reprodutíveis")

## 1. O que a engenharia de dados faz

In [ ]:
BASE = preparar("aula_10_01")

papeis = [
    ["Eng. de Software",  "sistemas que ATENDEM",   "API, latência, corretude"],
    ["Eng. de Dados",     "dados que INFORMAM",     "pipeline, qualidade, custo"],
    ["Analista de Dados", "responder perguntas",    "SQL, painel, narrativa"],
    ["Cientista de Dados", "prever e explicar",     "estatística, modelo"],
]
tabela(["PAPEL", "ENTREGA", "PREOCUPAÇÃO"], papeis, [20, 24, 32])

print("""
💭 A FRONTEIRA É BORRADA, E TUDO BEM.

   Numa empresa do tamanho da Aurora, você é os quatro. O que muda é o
   CHAPÉU que você usa em cada momento — e as perguntas que ele faz.

   Como engenheiro de SOFTWARE (M01–M09) você perguntou:
     "esta requisição responde em quanto tempo?"

   Como engenheiro de DADOS você pergunta:
     "este número está certo? de quando ele é? quanto custa mantê-lo?"

🎯 A pergunta nova é a do MEIO: "de quando ele é?"

   Em software, o dado é o presente. Em dados, ele tem IDADE — e a
   idade aceitável é uma decisão de negócio, não técnica.
""")

## 2. 🎯 OLTP vs OLAP — medindo a diferença

In [ ]:
# Os dados da Aurora
vendas = gerar_vendas(n=200_000, dias=365)
print(f"{len(vendas):,} vendas · {vendas['data'].min().date()} a "
      f"{vendas['data'].max().date()}")
print(f"memória: {tamanho(memoria(vendas))}\n")
print(vendas.head(4).to_string(index=False))

In [ ]:
# Montamos um banco relacional de verdade (o do M03/M05)
BANCO = BASE / "atlas.db"
conexao = sqlite3.connect(BANCO)

vendas_sql = vendas.copy()
vendas_sql["data"] = vendas_sql["data"].dt.tz_localize(None)
vendas_sql.to_sql("vendas", conexao, index=False, if_exists="replace")

# 🔑 Índice para a consulta TRANSACIONAL (M03)
conexao.execute("CREATE INDEX idx_pedido ON vendas(pedido_id)")
conexao.execute("CREATE INDEX idx_data ON vendas(data)")
conexao.commit()

print(f"banco: {tamanho(BANCO.stat().st_size)}")

In [ ]:
# ═══ Duas perguntas MUITO diferentes ═══
print("""
   OLTP — On-Line Transaction Processing
   ════════════════════════════════════
   "Qual é o pedido 142.857?"
   → toca 1 linha · precisa ser instantâneo · acontece 10.000×/minuto

   OLAP — On-Line Analytical Processing
   ═══════════════════════════════════
   "Qual o faturamento por categoria e canal, mês a mês, em 12 meses?"
   → toca TODAS as linhas · alguns segundos servem · acontece 20×/dia
""")

PERGUNTA_OLTP = "SELECT * FROM vendas WHERE pedido_id = 142857"
PERGUNTA_OLAP = """
    SELECT categoria, canal, strftime('%Y-%m', data) AS mes,
           SUM(quantidade * preco_unitario) AS receita
    FROM vendas
    WHERE status = 'pago'
    GROUP BY categoria, canal, mes
"""

_, ms_oltp = cronometrar(
    lambda: conexao.execute(PERGUNTA_OLTP).fetchall(), repeticoes=50)
_, ms_olap = cronometrar(
    lambda: conexao.execute(PERGUNTA_OLAP).fetchall(), repeticoes=3)

print(f"   OLTP (1 pedido)   : {ms_oltp:>9.3f} ms")
print(f"   OLAP (agregação)  : {ms_olap:>9.1f} ms")
print(f"\n   a analítica é {ms_olap / ms_oltp:,.0f}× mais cara")

> 🔴 **Agora imagine essa consulta rodando no banco de PRODUÇÃO.**
>
> Ela varre a tabela inteira. Enquanto isso, ela segura páginas em memória, gera I/O, e — dependendo do banco e do nível de isolamento — segura recursos que o checkout precisa.
>
> **Foi exatamente o que aconteceu com a Aurora:** o site ficou lento por 4 minutos porque alguém pediu um relatório.
>
> 🎯 **A separação OLTP/OLAP não é sobre tecnologia. É sobre não deixar a pergunta da diretoria competir com a compra do cliente.**

In [ ]:
caracteristicas = [
    ["Pergunta típica",   "qual é o pedido X?",       "quanto vendemos por Y?"],
    ["Linhas tocadas",    "1 a 100",                  "milhões"],
    ["Colunas tocadas",   "todas da linha",           "3 a 5 de muitas"],
    ["Latência aceitável", "milissegundos",           "segundos a minutos"],
    ["Frequência",        "milhares/segundo",         "dezenas/dia"],
    ["Escrita",           "constante",                "em lote"],
    ["Normalização",      "alta (3NF)",               "🔑 desnormalizado"],
    ["Armazenamento",     "por LINHA",                "🔑 por COLUNA"],
    ["Exemplos",          "PostgreSQL, MySQL",        "BigQuery, Redshift, DuckDB"],
]
tabela(["", "OLTP", "OLAP"], caracteristicas, [20, 26, 30])

## 3. 🎯 Por que colunar muda tudo

In [ ]:
print("""
   ARMAZENAMENTO POR LINHA (OLTP)
   ═══════════════════════════════
   [1|2026-01-05|NB-1000|Campinas|site|3|2599.90|pago]
   [2|2026-01-05|MO-1053|S.Paulo |app |1|1559.03|pago]

   Para somar `preco_unitario`, o banco precisa LER as linhas inteiras.
   Ele carrega cidade, canal, status — tudo que você não pediu.

   ARMAZENAMENTO POR COLUNA (OLAP)
   ═══════════════════════════════
   pedido_id      [1, 2, 3, ...]
   data           [2026-01-05, 2026-01-05, ...]
   preco_unitario [2599.90, 1559.03, ...]   ← lê SÓ isto
   status         [pago, pago, ...]

   Três ganhos, e todos se multiplicam:
     1. lê só as colunas pedidas
     2. valores iguais ficam juntos → comprime muito melhor
     3. tipo homogêneo → a CPU processa em lote (SIMD)
""")

# Provando o ganho de compressão
uma_coluna = vendas[["categoria"]]
print("A coluna `categoria` tem só 4 valores distintos, repetidos 200 mil vezes.\n")
print(f"   como texto solto : {tamanho(memoria(uma_coluna))}")
como_categoria = uma_coluna.astype("category")
print(f"   como 'category'  : {tamanho(memoria(como_categoria))}")
print(f"   redução          : "
      f"{(1 - memoria(como_categoria) / memoria(uma_coluna)) * 100:.1f}%")

In [ ]:
# ═══ A mesma pergunta, três motores ═══
if TEM_ARROW:
    PARQUET = BASE / "vendas.parquet"
    CSV = BASE / "vendas.csv"
    vendas.to_parquet(PARQUET, index=False)
    vendas.to_csv(CSV, index=False)

    print("O MESMO conjunto de dados, em dois formatos:\n")
    print(f"   CSV     : {tamanho(CSV.stat().st_size)}")
    print(f"   Parquet : {tamanho(PARQUET.stat().st_size)}")
    print(f"   redução : "
          f"{(1 - PARQUET.stat().st_size / CSV.stat().st_size) * 100:.1f}%")
    print("\n   💭 Mesmos dados. A diferença é só a ORGANIZAÇÃO.")
else:
    print("⚠️ pyarrow indisponível")

In [ ]:
if TEM_ARROW and TEM_DUCKDB:
    import duckdb

    def com_sqlite():
        return conexao.execute(PERGUNTA_OLAP).fetchall()

    def com_pandas():
        d = pd.read_parquet(PARQUET)
        d = d[d["status"] == "pago"].copy()
        d["receita"] = d["quantidade"] * d["preco_unitario"]
        d["mes"] = d["data"].dt.strftime("%Y-%m")
        return d.groupby(["categoria", "canal", "mes"], observed=True)["receita"].sum()

    def com_duckdb():
        return duckdb.sql(f"""
            SELECT categoria, canal, strftime(data, '%Y-%m') AS mes,
                   SUM(quantidade * preco_unitario) AS receita
            FROM '{PARQUET}'
            WHERE status = 'pago'
            GROUP BY categoria, canal, mes
        """).fetchall()

    print("A MESMA pergunta analítica, três motores:\n")
    comparar([("SQLite (linha, no disco)", com_sqlite),
              ("pandas (lê tudo p/ memória)", com_pandas),
              ("DuckDB sobre Parquet", com_duckdb)],
             repeticoes=3, rotulo="motor")
else:
    print("⚠️ duckdb ou pyarrow indisponível — saída de referência:")
    print("   SQLite            220 ms")
    print("   pandas             95 ms")
    print("   DuckDB             18 ms      [referência]")

> 🎯 **O DuckDB lê o mesmo arquivo Parquet e responde em uma fração do tempo.**
>
> Ele não é "mais rápido" por mágica — ele é **colunar e vetorizado**. Lê só as 5 colunas da consulta, ignora as outras 6, e processa em blocos.
>
> ⚠️ **E repare no resultado do pandas: ele foi o MAIS LENTO dos três.**
>
> Isso costuma surpreender. O motivo é que ele faz o oposto do DuckDB: carrega o arquivo **inteiro** para a memória — as 11 colunas, inclusive as 6 que a pergunta não usa — e só então filtra.
>
> 💭 **A lição não é "pandas é ruim".** É que a ferramenta certa depende da pergunta e do tamanho. Para 200 mil linhas e uma agregação, o DuckDB ganha. Para explorar, limpar e transformar interativamente, o pandas é imbatível — e é sobre isso a próxima aula.
>
> 💭 **E repare no que o DuckDB NÃO é:** um servidor. Ele roda dentro do seu processo Python, como o SQLite. Para a Aurora, isso significa análise de milhões de linhas **sem infraestrutura nenhuma** — e é por isso que ele aparece tanto neste módulo.
>
> ⚠️ **Mas não conclua que o Postgres é ruim.** Peça a esses três motores *"qual é o pedido 142.857?"* e o SQLite com índice ganha de longe. **São ferramentas para perguntas diferentes.**

## 4. Lake, Warehouse, Lakehouse

In [ ]:
arquiteturas = [
    ["Data Warehouse", "estruturado, schema-on-write", "caro, rígido, confiável"],
    ["Data Lake",      "cru, schema-on-read",          "barato, flexível, vira pântano"],
    ["Lakehouse",      "lake + camada transacional",   "o meio-termo atual"],
]
tabela(["", "O QUE É", "CARACTERÍSTICA"], arquiteturas, [18, 32, 32])

print("""
💭 A HISTÓRIA, EM TRÊS PARÁGRAFOS

   O WAREHOUSE veio primeiro: você modela o schema ANTES de gravar.
   Dado que não cabe no modelo não entra. Confiável, e caro — cada
   pergunta nova exige um projeto.

   O LAKE foi a reação: grave tudo, cru, em arquivos baratos, e
   decida o schema na hora de LER. Barato e flexível — e virou
   "pântano de dados" em muita empresa, porque ninguém sabia o que
   havia lá nem se dava para confiar.

   O LAKEHOUSE é a síntese: arquivos baratos (Parquet) + uma camada
   que dá transação, versionamento e schema (Delta, Iceberg).

🎯 E PARA A AURORA, HOJE?

   Nenhum dos três. Com 200 mil vendas por ano, "data lake" é
   arquitetura de empresa que não é a sua.

   O suficiente é: Parquet numa pasta (ou num bucket) + DuckDB.
   Isso responde qualquer pergunta da diretoria em segundos, custa
   quase nada, e cabe num script.

   ⚠️ Adotar arquitetura grande cedo demais é a forma mais comum de
      um time pequeno parar de entregar.
""")

> 💭 **Quando a Aurora vai precisar de mais?**
>
> | Sinal | O que muda |
> |-------|-----------|
> | Os dados não cabem na memória de uma máquina | Motor distribuído (Spark) ou nuvem |
> | Várias equipes consultando ao mesmo tempo | Warehouse gerenciado |
> | Precisa de dado em segundos, não horas | Streaming (aula 10_06) |
> | Auditoria exige histórico de versões | Lakehouse com Delta/Iceberg |
>
> **Nenhum desses sinais é "eu li um artigo sobre isso".** Cada um tem um custo operacional real — e a pergunta certa é sempre *"o que dói hoje?"*.

## 5. 🎯 As camadas — bronze, prata, ouro

In [ ]:
print("""
   ┌──── BRONZE (raw) ─────────────────────────────────────┐
   │ Exatamente como chegou. Nada corrigido.               │
   │ · CSV do ERP, resposta da API, dump do banco          │
   │ · 🔑 IMUTÁVEL — nunca se reescreve                     │
   └───────────────────────┬───────────────────────────────┘
                           ▼
   ┌──── PRATA (staging) ──────────────────────────────────┐
   │ Limpo, tipado, validado, deduplicado.                 │
   │ · uma linha por evento de negócio                     │
   │ · o que não passou vai para QUARENTENA                │
   └───────────────────────┬───────────────────────────────┘
                           ▼
   ┌──── OURO (final) ─────────────────────────────────────┐
   │ Agregado para consumo. Modelado para a pergunta.      │
   │ · faturamento por categoria/mês                       │
   │ · o que o painel e o relatório leem                   │
   └───────────────────────────────────────────────────────┘
""")

print("""🔑 POR QUE GUARDAR O BRONZE, SE ELE É SUJO?

   Porque daqui a três meses você vai descobrir um bug na sua
   limpeza. Com o bronze guardado, você reprocessa e conserta o
   histórico. Sem ele, o dado errado é o único que existe.

   🎯 A regra: TRANSFORMAÇÃO É CÓDIGO, e código tem bug. O dado cru é
      a sua chance de rodar a versão corrigida.

   💭 É a mesma ideia do M02: o Git guarda o histórico para você poder
      voltar. O bronze guarda a entrada para você poder recalcular.
""")

In [ ]:
# Montando as três camadas de verdade
LAGO = BASE / "lago"
for camada in ("bronze", "prata", "ouro", "quarentena"):
    (LAGO / camada).mkdir(parents=True)

# ── BRONZE: como chegou, particionado por data de ingestão ──
hoje = date(2026, 8, 13)
destino_bronze = LAGO / "bronze" / f"data_ingestao={hoje}"
destino_bronze.mkdir(parents=True)

# 🔑 Particionar por data de ingestão permite reprocessar UM dia.
if TEM_ARROW:
    vendas.to_parquet(destino_bronze / "vendas.parquet", index=False)

    # ── PRATA: limpo e tipado ──
    prata = vendas[vendas["status"] != "cancelado"].copy()
    prata["receita"] = (prata["quantidade"] * prata["preco_unitario"]).round(2)
    prata["margem"] = (
        prata["quantidade"] * (prata["preco_unitario"] - prata["custo_unitario"])
    ).round(2)
    prata["categoria"] = prata["categoria"].astype("category")
    prata["cidade"] = prata["cidade"].astype("category")
    prata.to_parquet(LAGO / "prata" / "vendas.parquet", index=False)

    # ── OURO: agregado para a pergunta ──
    ouro = (prata.assign(mes=prata["data"].dt.to_period("M").astype(str))
            .groupby(["mes", "categoria", "canal"], observed=True)
            .agg(pedidos=("pedido_id", "nunique"),
                 itens=("quantidade", "sum"),
                 receita=("receita", "sum"),
                 margem=("margem", "sum"))
            .reset_index())
    ouro.to_parquet(LAGO / "ouro" / "faturamento_mensal.parquet", index=False)

    for camada in ("bronze", "prata", "ouro"):
        arquivos = list((LAGO / camada).rglob("*.parquet"))
        total = sum(a.stat().st_size for a in arquivos)
        linhas = {"bronze": len(vendas), "prata": len(prata), "ouro": len(ouro)}[camada]
        print(f"   {camada:<10} {linhas:>8,} linhas   {tamanho(total):>12}")

    print(f"\n   🎯 do bronze ao ouro: {len(vendas):,} → {len(ouro):,} linhas")
    print(f"      ({len(vendas) / len(ouro):,.0f}× menos, e a pergunta responde igual)")
else:
    print("⚠️ pyarrow indisponível")

In [ ]:
if TEM_ARROW:
    print("A camada OURO — o que a diretoria enxerga:\n")
    print(ouro.sort_values("receita", ascending=False).head(8).to_string(index=False))

    print(f"\n💡 Responder 'faturamento de Notebooks no site em junho' agora")
    print(f"   custa uma leitura de {tamanho((LAGO / 'ouro' / 'faturamento_mensal.parquet').stat().st_size)},")
    print("   não uma varredura de 200 mil linhas no banco de produção.")

## 6. Modelagem dimensional

In [ ]:
print("""
   ESQUEMA ESTRELA

                  ┌──────────────┐
                  │ dim_produto  │
                  │ sku (PK)     │
                  │ nome         │
                  │ categoria    │
                  └──────┬───────┘
                         │
   ┌──────────────┐  ┌───▼──────────────┐  ┌──────────────┐
   │ dim_tempo    │  │  fato_vendas     │  │ dim_local    │
   │ data (PK)    ├──┤  data (FK)       ├──┤ cidade (PK)  │
   │ ano, mes     │  │  sku (FK)        │  │ uf, regiao   │
   │ dia_semana   │  │  cidade (FK)     │  └──────────────┘
   │ e_feriado    │  │  quantidade   ←── MÉTRICAS
   └──────────────┘  │  receita      ←── (o que se soma)
                     │  margem       ←──
                     └──────────────────┘

   FATO       o que aconteceu · muitas linhas · números que se SOMAM
   DIMENSÃO   o contexto · poucas linhas · atributos que se FILTRA
""")

print("""💭 POR QUE DESNORMALIZAR, SE O M03 ENSINOU A NORMALIZAR?

   Porque o objetivo mudou.

   No OLTP, normalizar evita ANOMALIA DE ATUALIZAÇÃO: o preço mora num
   lugar só, então não há como ficar inconsistente.

   No OLAP, o dado é HISTÓRICO — ele não é atualizado. E cada JOIN
   custa caro quando você varre milhões de linhas.

   🎯 A regra: normalize onde há ESCRITA. Desnormalize onde há LEITURA
      analítica.

   ⚠️ E tem um detalhe que quase todo mundo erra: a tabela de fato
      guarda o preço DO MOMENTO DA VENDA, não o preço atual do
      produto. Se você fizer JOIN com a dimensão para pegar o preço,
      o faturamento do ano passado muda quando alguém reajusta a
      tabela — e o número que você apresentou ao investidor deixa de
      bater.
""")

In [ ]:
if TEM_ARROW:
    # A dimensão tempo — a que mais rende
    datas = pd.date_range("2025-08-01", "2026-08-31", freq="D")
    dim_tempo = pd.DataFrame({
        "data": datas,
        "ano": datas.year,
        "mes": datas.month,
        "dia": datas.day,
        "trimestre": datas.quarter,
        "dia_semana": datas.dayofweek,
        "nome_dia": datas.day_name(),
        "fim_de_semana": datas.dayofweek >= 5,
    })
    dim_tempo.to_parquet(LAGO / "ouro" / "dim_tempo.parquet", index=False)

    print(f"dim_tempo: {len(dim_tempo)} linhas\n")
    print(dim_tempo.head(4).to_string(index=False))

    print("""
💡 POR QUE UMA TABELA SÓ COM DATAS?

   Porque ela responde perguntas que a data sozinha não responde:
   "vendemos mais no fim de semana?", "como foi o 2º trimestre?",
   "e nos feriados?"

   Sem ela, cada consulta reimplementa o calendário — e alguém sempre
   erra a semana do ano ou o feriado móvel.
""")

    resposta = (prata.assign(fds=prata["data"].dt.dayofweek >= 5)
                .groupby("fds", observed=True)["receita"].agg(["count", "sum"]))
    resposta.index = ["dia útil", "fim de semana"]
    print("Exemplo de pergunta que a dimensão tempo responde:\n")
    print(resposta.to_string())

## 7. Batch vs streaming

In [ ]:
comparacao = [
    ["Quando roda",     "de hora em hora, de madrugada", "o tempo todo"],
    ["Latência",        "minutos a horas",                "segundos"],
    ["Complexidade",    "baixa",                          "🔶 alta"],
    ["Reprocessar",     "✅ trivial",                     "🔴 difícil"],
    ["Custo",           "baixo",                          "alto (fica no ar)"],
    ["Ferramentas",     "cron, Airflow",                  "Kafka, Flink"],
]
tabela(["", "BATCH (lote)", "STREAMING"], comparacao, [18, 32, 26])

print("""
🎯 A PERGUNTA QUE DECIDE — E NÃO É TÉCNICA:

   "Qual é o CUSTO de a informação ter 1 hora de atraso?"

   Para o faturamento por categoria: nenhum. Ninguém toma decisão
   diferente com o número de uma hora atrás.

   Para detecção de fraude no cartão: enorme. Uma hora depois, o
   dinheiro já saiu.

⚠️ E na dúvida, comece em BATCH.

   Streaming multiplica a complexidade de tudo: ordem das mensagens,
   entrega duplicada, janelas de tempo, estado distribuído,
   reprocessamento. É uma escolha que se paga todo dia.

   💭 Muita empresa monta streaming e usa o resultado num painel que
      alguém olha uma vez por dia.
""")

## 8. O custo — o que ninguém conta no começo

In [ ]:
print("""
🔴 O CUSTO DE UM PIPELINE NÃO É ESCREVÊ-LO. É MANTÊ-LO.

   O que aparece depois:

   · o fornecedor mudou o CSV sem avisar          → quebra semanal
   · o pipeline falhou às 3h e ninguém viu        → dado velho no painel
   · rodou duas vezes e duplicou tudo             → 🔴 número errado
   · alguém precisa reprocessar 6 meses           → e não dá
   · duas métricas com o mesmo nome e valores diferentes
   · ninguém sabe de onde vem aquele número

🎯 E O ÚLTIMO É O PIOR.

   Um número em que ninguém confia é pior do que nenhum número: a
   empresa toma a decisão pelo instinto de qualquer forma, mas agora
   com uma reunião extra para discutir a planilha.
""")

custos = [
    ["Idempotência",   "rodar 2× dá o mesmo resultado",   "aula 10_05"],
    ["Reprocessável",  "recalcular 6 meses é um comando", "camada bronze"],
    ["Observável",     "você sabe que falhou, e onde",    "M09"],
    ["Validado",       "dado ruim é barrado, não some",   "aula 10_05"],
    ["Documentado",    "de onde vem cada número",         "linhagem"],
    ["Testado",        "a transformação tem teste",       "M07"],
]
tabela(["PROPRIEDADE", "O QUE SIGNIFICA", "ONDE"], custos, [18, 36, 20])

print("""
💭 OLHE A COLUNA DA DIREITA.

   Metade das propriedades de um bom pipeline você já construiu — em
   módulos que não falavam de dados.

   🎯 Um pipeline de dados É um sistema de software. Ele só tem duas
      diferenças: o dado é o produto, e ele roda sozinho de madrugada.
""")

## 🔧 Prática guiada — o diagnóstico da Aurora

In [ ]:
if TEM_ARROW and TEM_DUCKDB:
    print("A pergunta original da diretora, agora na camada certa:\n")

    def pelo_banco_de_producao():
        return conexao.execute(PERGUNTA_OLAP).fetchall()

    def pela_camada_ouro():
        return duckdb.sql(
            f"SELECT * FROM '{LAGO / 'ouro' / 'faturamento_mensal.parquet'}'"
        ).fetchall()

    medidos = comparar([
        ("banco de PRODUÇÃO (🔴 trava o site)", pelo_banco_de_producao),
        ("camada OURO (já calculada)", pela_camada_ouro),
    ], repeticoes=3, rotulo="onde perguntar")

    print(f"\n🎯 E o ganho maior nem é o tempo — é que a segunda consulta")
    print("   NÃO TOCA no banco que atende o cliente.")

In [ ]:
# O plano para a Aurora, escrito
PLANO = BASE / "PLANO_DADOS.md"
PLANO.write_text("""# Plano de dados da Aurora

## O problema
1. Relatório pesado no banco de produção derruba o site
2. Decisões tomadas com dado de uma semana atrás

## A resposta — proporcional ao tamanho da empresa

### Não vamos fazer (ainda)
- ❌ Data lake com camadas em nuvem
- ❌ Spark / cluster
- ❌ Streaming
- ❌ Warehouse gerenciado

**Motivo:** 200 mil vendas/ano cabem na memória de um notebook. Cada
uma dessas escolhas custa operação e tempo que a Aurora não tem.

### Vamos fazer
- ✅ Extração diária do Postgres → Parquet (camada bronze)
- ✅ Limpeza e tipagem → prata
- ✅ Agregações do negócio → ouro
- ✅ DuckDB para consultar o ouro
- ✅ Agendamento diário, com alerta em caso de falha

**Custo:** um script, uma pasta e um agendamento.

## Como saberemos que precisamos de mais
- [ ] Os dados não cabem mais em memória
- [ ] Mais de uma equipe consultando ao mesmo tempo
- [ ] Alguma decisão exige dado de minutos, não de horas
- [ ] Auditoria exige histórico versionado

## Princípios
1. **Bronze é imutável.** Nunca se reescreve.
2. **Rodar duas vezes dá o mesmo resultado.**
3. **Dado ruim vai para quarentena**, não some e não entra.
4. **Toda métrica tem uma definição escrita** e um dono.
5. **O pipeline avisa quando falha** (M09).
""", encoding="utf-8")

print(PLANO.read_text(encoding="utf-8"))

In [ ]:
conexao.close()
print("Estrutura criada:\n")
for caminho in sorted(LAGO.rglob("*")):
    if caminho.is_file():
        print(f"   {caminho.relative_to(BASE)}  ({tamanho(caminho.stat().st_size)})")

## 📝 Exercícios

**E1.** Explique, com as suas palavras, por que a consulta analítica derrubou o site da Aurora.

**E2.** Escreva três perguntas OLTP e três OLAP sobre o Atlas. Justifique a classificação.

**E3.** Meça a diferença entre uma consulta por chave primária e uma agregação completa no seu banco.

**E4.** 🔴 Converta uma coluna de texto repetido para `category` e meça a redução de memória.

**E5.** Salve o mesmo `DataFrame` em CSV e Parquet. Compare tamanho e tempo de leitura.

**E6.** Explique por que o formato colunar comprime melhor. Dê um exemplo com os dados da Aurora.

**E7.** Argumente, por escrito, se a Aurora precisa de um data lake hoje. Use números.

**E8.** 🔑 Explique por que a camada bronze é imutável, com um cenário concreto de bug de transformação.

**E9.** Monte as três camadas para os seus próprios dados. Meça a redução de linhas do bronze ao ouro.

**E10.** Desenhe um esquema estrela para os pedidos da Aurora. Identifique fato, dimensões e métricas.

**E11.** 🔴 Explique por que a tabela de fato guarda o preço do momento da venda. O que quebra se ela fizer JOIN com o preço atual?

**E12.** Crie uma `dim_tempo` e use-a para responder "vendemos mais no fim de semana?".

**E13.** Para cada dado do Atlas, decida batch ou streaming. Justifique pelo custo do atraso.

**E14.** Liste as seis propriedades de um bom pipeline e diga quais o seu Atlas já tem.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

## 📋 Cola de referência

```
═══ OLTP vs OLAP ═══
OLTP  1 linha · ms · milhares/s · normalizado · por LINHA
OLAP  milhões · segundos · dezenas/dia · desnormalizado · por COLUNA
🔴 nunca rode analítica no banco que atende o cliente

═══ Por que colunar ═══
1. lê só as colunas pedidas
2. valores iguais juntos → comprime muito melhor
3. tipo homogêneo → processamento vetorizado

═══ Camadas ═══
BRONZE  como chegou · 🔑 IMUTÁVEL · particionado por data de ingestão
PRATA   limpo, tipado, validado · quarentena para o que falhou
OURO    agregado para a pergunta · o que o painel lê

🔑 guarde o bronze: quando a transformação tiver bug, você reprocessa

═══ Modelagem ═══
FATO       o evento · muitas linhas · métricas que se SOMAM
DIMENSÃO   o contexto · poucas linhas · atributos que se FILTRA
🔴 o fato guarda o preço DO MOMENTO, não faz JOIN para pegar o atual

═══ Batch vs streaming ═══
a pergunta: "qual o custo de 1 hora de atraso?"
⚠️ na dúvida, batch — streaming multiplica a complexidade de tudo

═══ Ferramentas para o tamanho da Aurora ═══
Parquet numa pasta + DuckDB   → responde tudo, custa quase nada
```

## ✅ Checklist de saída

- [ ] Sei por que analítica no banco de produção derruba o site
- [ ] Distingo pergunta OLTP de OLAP
- [ ] 🎯 **Sei por que colunar é ordens de grandeza melhor para agregação**
- [ ] Sei que o inverso também vale: linha ganha na busca por chave
- [ ] Conheço lake, warehouse e lakehouse
- [ ] 💭 **Sei argumentar que a Aurora não precisa de nenhum deles hoje**
- [ ] Sei o papel de cada camada
- [ ] 🔑 **Sei por que o bronze é imutável**
- [ ] Distingo fato de dimensão
- [ ] 🔴 **Sei por que o fato guarda o preço do momento da venda**
- [ ] Sei quando uma `dim_tempo` compensa
- [ ] Uso o custo do atraso para decidir batch ou streaming
- [ ] Conheço as seis propriedades de um pipeline sustentável

---

### ➡️ Próxima aula

**`10_02_Pandas_Essencial.ipynb`** — A ferramenta que você vai usar todo dia. E as armadilhas que fazem o número sair errado sem nenhum erro aparecer.